# Tutoriel K-ABENA — MLP / ANN (niveau 1 : notebook)
**Plateformes couvertes** : PyTorch **et** TensorFlow/Keras (deux sections indépendantes — exécutez celle de votre stack).
Prérequis GPU : aucun (MLP petit). `pip install kabena torch` ou `pip install kabena tensorflow`.

## Section A — PyTorch (2 lignes : `KabenaTorch()` puis `kb.reduce(losses)`)

In [ ]:
import torch, torch.nn as nn
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from kabena.integrations.torch import KabenaTorch

D = load_digits()
Xtr, Xte, ytr, yte = train_test_split(D.data, D.target, test_size=0.25, random_state=0, stratify=D.target)
sc = StandardScaler().fit(Xtr)
Xtr_t = torch.tensor(sc.transform(Xtr), dtype=torch.float32)
ytr_t = torch.tensor(ytr)
model = nn.Sequential(nn.Linear(64, 32), nn.Tanh(), nn.Linear(32, 10))
opt = torch.optim.SGD(model.parameters(), lr=0.5)
crit = nn.CrossEntropyLoss(reduction="none")     # IMPORTANT : pertes par échantillon

kb = KabenaTorch(seed=0)                          # <= LIGNE 1
for epoch in range(40):
    losses = crit(model(Xtr_t), ytr_t)
    loss = kb.reduce(losses, y=ytr_t)             # <= LIGNE 2 (masque + poids HT)
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    acc = (model(torch.tensor(sc.transform(Xte), dtype=torch.float32)).argmax(1).numpy() == yte).mean()
print(f"accuracy test = {acc:.4f} | gain = {kb.last_gain_*100:.1f}%")

## Section B — TensorFlow/Keras (2 lignes : callback + `sample_weight`)

In [ ]:
import numpy as np, tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from kabena.integrations.keras import KabenaKeras

D = load_digits()
Xtr, Xte, ytr, yte = train_test_split(D.data, D.target, test_size=0.25, random_state=0, stratify=D.target)
sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

model = keras.Sequential([keras.layers.Dense(32, activation="tanh", input_shape=(64,)),
                          keras.layers.Dense(10, activation="softmax")])
model.compile(optimizer="sgd", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def per_sample_loss(m, X, y):
    p = m.predict(X, verbose=0)
    return -np.log(np.clip(p[np.arange(len(y)), y], 1e-9, 1))

kb = KabenaKeras(model, Xtr, ytr, per_sample_loss, seed=0)          # <= LIGNE 1
model.fit(Xtr, ytr, epochs=40, verbose=0,
          callbacks=[kb], sample_weight=kb.weights)                 # <= LIGNE 2
print("accuracy test =", model.evaluate(Xte, yte, verbose=0)[1], "| gain =", kb.last_gain_)

**Note honnête (Limitation L4 du preprint)** : ces sections DL n'ont pas été exécutées dans
l'environnement de validation du package (CPU/NumPy only) — le code est vérifié syntaxiquement
et suit les intégrations testées ; signalez tout écart via GitHub Issues.